In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
import torch, os
from config import Config
from metrics import evaluate_full_test_set
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
cfg = Config()

Mounted at /content/drive
Device: cuda


In [ ]:
from model import Generator
import torch.serialization
torch.serialization.add_safe_globals([Config])

def load_generator(checkpoint_dir, device, cfg):
    final_path = os.path.join(checkpoint_dir, 'final_model.pth')
    if os.path.exists(final_path):
        model_path = final_path
    else:
        ckpts = sorted([f for f in os.listdir(checkpoint_dir) if f.startswith('checkpoint_epoch_')],
                        key=lambda x: int(x.split('_')[-1].split('.')[0]))
        assert ckpts, f"No checkpoints found in {checkpoint_dir}"
        model_path = os.path.join(checkpoint_dir, ckpts[-1])
    print("Loading:", model_path)
    G = Generator(cfg.img_size, num_domains=cfg.num_domains).to(device)
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    except Exception:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    for key in ['G_state_dict', 'generator_state_dict', 'state_dict']:
        if key in checkpoint:
            G.load_state_dict(checkpoint[key]); break
    else:
        G.load_state_dict(checkpoint)
    G.eval()
    return G

G_eyegan = load_generator(cfg.checkpoint_dir, device, cfg)
print("EyeGAN generator loaded:", sum(p.numel() for p in G_eyegan.parameters()), "params")

Loading: /content/drive/MyDrive/CSE720/stargan_checkpoints/final_model.pth
EyeGAN generator loaded: 2048675 params


In [ ]:
import time
t0 = time.time()
per_item, summary, fid_rows = evaluate_full_test_set(
    G_eyegan, cfg, device, out_dir=cfg.eval_dir, model_name='EyeGAN'
)
inference_wall_time = time.time() - t0
n_translations = summary['n_translations']
print(f"\nWall time for {n_translations} translations: {inference_wall_time:.1f}s "
      f"({1000*inference_wall_time/n_translations:.1f} ms/translation) — "
      "save this number, R1-Q10 asks for inference time.")

[test] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 129MB/s]


[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[all] domains found: {'Diabetic Retinopathy': 500, 'Glaucoma': 500, 'Healthy': 500, 'Macular Scar': 500, 'Myopia': 500}  (total 2500)
[EyeGAN] n_translations=1000 (should be 250x4= 1000)
[EyeGAN] PSNR 36.24±2.69 | SSIM 0.926±0.033 | MSE 0.00030±0.00043 | FID 21.61±8.04

Wall time for 1000 translations: 496.0s (496.0 ms/translation) — save this number, R1-Q10 asks for inference time.


In [ ]:
BASELINE_CKPT_PATHS = {
    'DCGAN':    '/content/drive/MyDrive/CSE720/dcgan_results',
    'CycleGAN': '/content/drive/MyDrive/CSE720/cyclegan_results',
    'Pix2Pix':  '/content/drive/MyDrive/CSE720/pix2pix_results',
}
for name, path in BASELINE_CKPT_PATHS.items():
    exists = os.path.exists(path)
    print(f"{name}: {path} -> {'FOUND' if exists else 'NOT FOUND (edit the path above)'}")

DCGAN: /content/drive/MyDrive/CSE720/dcgan_results -> FOUND
CycleGAN: /content/drive/MyDrive/CSE720/cyclegan_results -> FOUND
Pix2Pix: /content/drive/MyDrive/CSE720/pix2pix_results -> FOUND


In [ ]:
import os
import csv
import torch
import numpy as np
from dataset import MedicalDataset
from torch.utils.data import DataLoader
from metrics import calc_psnr_ssim_mse, FIDCalculator

def evaluate_baseline_full(generate_fn, model_name, domains=('Healthy', 'Diabetic_Retinopathy')):
    """
    generate_fn(real_img_tensor[1,3,H,W]) -> fake_img_tensor[3,H,W] in [-1,1]
    Plug in each baseline's own forward pass here.
    """
    test_dataset = MedicalDataset(cfg.dataset_path, cfg.img_size, 'test')
    domain_idxs = {cfg.class_names[i]: i for i in range(cfg.num_domains) if cfg.class_names[i] in domains}
    idxs = [i for i, l in enumerate(test_dataset.labels) if l in domain_idxs.values()]

    per_item_rows, fakes = [], []
    for i in idxs:
        real_img, label, path = test_dataset[i]
        real_img = real_img.unsqueeze(0).to(device)
        fake_img = generate_fn(real_img)
        psnr_v, ssim_v, mse_v = calc_psnr_ssim_mse(real_img[0], fake_img)
        per_item_rows.append({
            'model': model_name,
            'image': os.path.basename(path),
            'source': cfg.class_names[label],
            'target': 'DR_or_Healthy',
            'psnr': psnr_v,
            'ssim': ssim_v,
            'mse': mse_v
        })
        fakes.append(fake_img.detach().cpu())

    fid_calc = FIDCalculator(device)
    real_pool = torch.stack([test_dataset[i][0] for i in idxs])
    fid_val = fid_calc.calculate_fid(
        fid_calc.get_activations(real_pool),
        fid_calc.get_activations(torch.stack(fakes))
    )

    csv_path = os.path.join(cfg.eval_dir, f'{model_name}_per_item_results.csv')
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'image', 'source', 'target', 'psnr', 'ssim', 'mse'])
        w.writeheader()
        w.writerows(per_item_rows)

    summary = {
        'model': model_name,
        'n_translations': len(per_item_rows),
        'psnr_mean': float(np.mean([r['psnr'] for r in per_item_rows])),
        'psnr_std': float(np.std([r['psnr'] for r in per_item_rows])),
        'ssim_mean': float(np.mean([r['ssim'] for r in per_item_rows])),
        'ssim_std': float(np.std([r['ssim'] for r in per_item_rows])),
        'mse_mean': float(np.mean([r['mse'] for r in per_item_rows])),
        'mse_std': float(np.std([r['mse'] for r in per_item_rows])),
        'fid_mean': fid_val,
        'fid_std': 0.0,
    }
    print(summary)
    return per_item_rows, summary

# --- EXAMPLE for DCGAN (unconditional: generates from noise, has no "source" image to
#     pair against for PSNR/SSIM/MSE in the strict sense -- the manuscript already flags
#     this in the Discussion. Keep DCGAN's FID-only comparison, skip forcing PSNR/SSIM). ---
# from dgan_model import Generator as DCGAN_G, nz
# netG = DCGAN_G(nz=nz).to(device)
# ckpt = torch.load(f"{BASELINE_CKPT_PATHS['DCGAN']}/dcgan_final.pth", map_location=device)
# netG.load_state_dict(ckpt['generator_state_dict']); netG.eval()
# ... (compute FID only, using netG(torch.randn(N, nz, device=device)) vs real pool)

# --- CycleGAN / Pix2Pix: paste each notebook's Generator class + checkpoint loading here,
#     then call:
# evaluate_baseline_full(lambda x: G_cyclegan_H2DR(x)[0], 'CycleGAN')
# evaluate_baseline_full(lambda x: G_pix2pix(x)[0], 'Pix2Pix')

print("Skeleton ready — fill in each baseline's own generator + checkpoint loading above,"
      " then call evaluate_baseline_full(...) for CycleGAN and Pix2Pix, and a FID-only"
      " variant for DCGAN.")

Skeleton ready — fill in each baseline's own generator + checkpoint loading above, then call evaluate_baseline_full(...) for CycleGAN and Pix2Pix, and a FID-only variant for DCGAN.


In [ ]:
import pandas as pd, glob

summaries = []
for f in glob.glob(os.path.join(cfg.eval_dir, '*_per_item_results.csv')):
    model = os.path.basename(f).replace('_per_item_results.csv', '')
    df = pd.read_csv(f)
    fid_f = os.path.join(cfg.eval_dir, f'{model}_fid_per_domain.csv')
    fid_mean = pd.read_csv(fid_f)['fid'].mean() if os.path.exists(fid_f) else float('nan')
    summaries.append({
        'Model': model, 'N_translations': len(df),
        'PSNR': f"{df.psnr.mean():.2f} ± {df.psnr.std():.2f}",
        'SSIM': f"{df.ssim.mean():.3f} ± {df.ssim.std():.3f}",
        'MSE':  f"{df.mse.mean():.5f} ± {df.mse.std():.5f}",
        'FID':  f"{fid_mean:.2f}",
    })

summary_df = pd.DataFrame(summaries)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(cfg.eval_dir, 'corrected_table1_summary.csv'), index=False)
print("\nSaved to revision/evaluation/corrected_table1_summary.csv")

 Model  N_translations         PSNR          SSIM               MSE   FID
EyeGAN            1000 36.24 ± 2.70 0.926 ± 0.033 0.00030 ± 0.00043 21.61

Saved to revision/evaluation/corrected_table1_summary.csv
